# Update dfs w/ new variables, column names etc. 

- instead of re-running dfmaker for now
- but this should be merged into makedf at some point

In [1]:
import pandas as pd
import numpy as np
from os import path

import sys
# sys.path.append('../../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.categories import *
from pyanalib.variable_calculator import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
# also turn off NaturalNameWarning (from PyTables)
import tables
warnings.filterwarnings("ignore", category=tables.NaturalNameWarning)

In [2]:
from makedf.g4syst import g4_systematics
from makedf.mcstat import get_MCstat_unc

In [3]:
def add_opening_angle(df, truth=False, nu=False):
    if nu:
        opening_angle = (
            df.mc.mu.dir[["x", "y", "z"]].values * df.mc.p.dir[["x", "y", "z"]].values
        ).sum(axis=1)

        df["theta_mu_p"] = np.arccos(opening_angle) * 180. / np.pi

    elif truth:
        opening_angle = (
            df.mc.mu.dir[["x", "y", "z"]].values * df.mc.p.dir[["x", "y", "z"]].values
        ).sum(axis=1)

        df["mc_theta_mu_p"] = np.arccos(opening_angle) * 180. / np.pi


    else:
        opening_angle = (
            df.mu.pfp.trk.dir[["x", "y", "z"]].values * df.p.pfp.trk.dir[["x", "y", "z"]].values
        ).sum(axis=1)

        df["theta_mu_p"] = np.arccos(opening_angle) * 180. / np.pi
    return df

In [4]:
def add_track_direction(df):
    print("adding track direction")
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","x")] = df.mu.pfp.trk.truth.p.genp.x / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","y")] = df.mu.pfp.trk.truth.p.genp.y / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","z")] = df.mu.pfp.trk.truth.p.genp.z / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","x")] = df.p.pfp.trk.truth.p.genp.x / df.p.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","y")] = df.p.pfp.trk.truth.p.genp.y / df.p.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","z")] = df.p.pfp.trk.truth.p.genp.z / df.p.pfp.trk.truth.p.totp
    return df

tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]

## Modify evtdf

In [23]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
# file_dir="/pnfs/sbnd/scratch/users/munjung/xsec/2025Spring_v10_06_00_09"
sub_dir="MC"
sample_dir="BNB_cosmics"

In [ ]:
# modify evtdf

# for tag in generate_tags("bl"):
for tag in ["GiBUU-sel_all"]:
    print("file tag:", tag)
    filename = path.join(file_dir, sub_dir, sample_dir, f"{tag}.df")
    this_split_df = pd.read_hdf(filename, key="split")
    this_n_split = this_split_df.n_split.iloc[0]

    with pd.HDFStore(filename, mode='r') as store:
        keys = store.keys() 
        pass
        # print("Keys:", keys)

    for i in range(this_n_split):
        evt_key = f"evt_{i}"
        hdr_key = f"hdr_{i}"
        this_df = pd.read_hdf(filename, key=evt_key)
        this_hdr_df = pd.read_hdf(filename, key=hdr_key)

        # ====== add stuff to the df ======
        try:
            # evetnt categories
            this_df.loc[:,'topo_categ'] = get_topo_category(this_df)
            this_df.loc[:,'genie_categ'] = get_genie_category(this_df)

            # add track direction
            this_df = add_track_direction(this_df)

            # opening angle
            this_df = add_opening_angle(this_df)
            this_df = add_opening_angle(this_df, truth=True)


            # add tki
            slc_mudf = this_df.mu.pfp.trk.truth.p
            slc_pdf = this_df.p.pfp.trk.truth.p
            slc_P_mu_col = pad_column_name(("totp",), slc_mudf)
            slc_P_p_col = pad_column_name(("totp",), slc_pdf)
            tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
            for var_name in tki_var_names:
                this_df = multicol_add(this_df, tki_reco[var_name].rename("mc_" + var_name))

        except:
            # raise ValueError(f"Error for {evt_key}")
            print(f"Error for {evt_key}")
            continue
        # ==================================

        # overwrite df with the same structure as the original df
        with pd.HDFStore(filename, mode='r+') as store:
            store.put(evt_key, this_df)

file tag: GiBUU-sel_all


AttributeError: 'DataFrame' object has no attribute 'mu'

## Modify nudf

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
sub_dir="MC"
sample_dir="BNB_cosmics/genie_wgts-Other"

for tag in generate_tags("bl"):
    tag += ""
    print("file tag:", tag)
    filename = path.join(file_dir, sub_dir, sample_dir, f"{tag}.df")
    this_split_df = pd.read_hdf(filename, key="split")
    this_n_split = this_split_df.n_split.iloc[0]

    with pd.HDFStore(filename, mode='r') as store:
        keys = store.keys() 
        # print("Keys:", keys)

    for i in range(this_n_split):
        mcnu_key = f"mcnu_{i}"
        this_nudf = pd.read_hdf(filename, key=mcnu_key)

        # ====== add stuff to the df ======
        try:
            # add multiindex column index "mc" so that branch names match evt_df
            this_nudf.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in this_nudf.columns])     # match # of column levels

            # event categories
            this_nudf.loc[:,'topo_categ'] = get_topo_category(this_nudf)
            this_nudf.loc[:,'genie_categ'] = get_genie_category(this_nudf)

            # opening angle
            this_nudf = add_opening_angle(this_nudf, nu=True)

            # tki
            tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]
            mc_mudf = this_nudf.mc.mu
            mc_pdf = this_nudf.mc.p
            mc_P_mu_col = pad_column_name(("totp",), mc_mudf)
            mc_P_p_col = pad_column_name(("totp",), mc_pdf)
            tki_mc = get_cc1p0pi_tki(mc_mudf, mc_pdf, mc_P_mu_col, mc_P_p_col)
            for var_name in tki_var_names:
                this_nudf = multicol_add(this_nudf, tki_mc[var_name].rename("{}".format(var_name)))

        except:
            # raise ValueError(f"Error for {evt_key}")
            print(f"Error for {mcnu_key}")
            continue
        # ==================================

        # overwrite df with the same structure as the original df
        with pd.HDFStore(filename, mode='r+') as store:
            store.put(mcnu_key, this_nudf)